# Building a CX Agent for Banking - Meridian Bank

**Business setting.** Meridian Bank handles a high volume of customer contacts. Experienced agents resolve them manually today. We will build an AI agent, Mia, that produces a first-pass resolution across three journeys: charge disputes, credit card applications, and loan enquiries.

The agent will: receive a customer request, decide which tools it needs, use them dynamically, loop until it has enough evidence, produce a structured outcome, and route through governance before any external action.

## Two frameworks this build follows

**1. The AI Agents Modules pyramid** - five layers, bottom to top:

| Layer | What it is | In this notebook |
| --- | --- | --- |
| Model | LLM reasoning and generation | Claude (claude-sonnet-4-6) via LangChain |
| Context | Internal knowledge, documents, databases | mock customers, transactions, policy KB |
| Tools | APIs, search, business actions | verify, read, dispute, escalate tools |
| Orchestration | State, routing, loops, approvals | LangGraph loop plus confidence routing |
| Governance | Security, access control, auditability | identity gate plus immutable audit trail |

**2. The basic agent workflow** - a user gives a business task; the agent interprets the goal; it loops until it has enough evidence (decide a tool, run it, feed the result back into state, decide if more is needed); it produces a structured output; then, before any external action, it goes through review.

**Our three design specifications** map onto the top two layers:

- Audit trail (reference id, timestamp, location, category, confidence, action, conversation id) maps to Governance.
- Verify before acting maps to the Orchestration approval gate, enforced inside every tool.
- Confidence routing (low equals escalate; medium equals triage then escalate; high equals resolve at agent) maps to Orchestration.

In [ ]:
%pip install -q langchain langgraph langchain-anthropic langchain-community pydantic pandas

In [ ]:
import os, json, uuid
from getpass import getpass
from datetime import datetime, timezone
from typing_extensions import Annotated, TypedDict
import pandas as pd
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
NL = chr(10)

## Layer 1 - Model

The reasoning engine. Everything else exists to feed it good context and to control what it is allowed to do. We read the key from a prompt so it is never hard-coded in the notebook. This build uses your Claude (Anthropic) API key.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Enter your Anthropic API key: ')
MODEL_NAME = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=2048)
print('Model ready:', MODEL_NAME)

## Layer 2 - Context

The agent is only as good as the business context it can reach. In production these are core-banking, card-processor, and knowledge systems. Here we mock a small slice so the whole thing runs end to end with no external dependencies.

In [ ]:
CUSTOMERS = {
  'CUST-88102': {'customer_id':'CUST-88102','name':'Daniel Okafor','dob':'1989-04-12','home_location':'Manchester, UK','phone_last4':'7741','segment':'mass_affluent','accounts':[{'account_id':'ACC-5567','type':'current','currency':'GBP','balance':4280.55,'status':'active'}],'cards':[{'card_id':'CARD-3392','type':'credit','last4':'9920','status':'active'}]},
  'CUST-88210': {'customer_id':'CUST-88210','name':'Aisha Rahman','dob':'1995-11-03','home_location':'Birmingham, UK','phone_last4':'2210','segment':'young_professional','accounts':[{'account_id':'ACC-7781','type':'current','currency':'GBP','balance':1190.10,'status':'active'}],'cards':[{'card_id':'CARD-6650','type':'debit','last4':'3082','status':'active'}]},
  'CUST-88345': {'customer_id':'CUST-88345','name':'George Whitfield','dob':'1972-06-28','home_location':'Leeds, UK','phone_last4':'5532','segment':'premier','accounts':[{'account_id':'ACC-9001','type':'current','currency':'GBP','balance':9850.00,'status':'active'}],'cards':[{'card_id':'CARD-1180','type':'credit','last4':'7705','status':'active'}]},
}
TRANSACTIONS = {
  'TXN-500021': {'txn_id':'TXN-500021','customer_id':'CUST-88102','card_id':'CARD-3392','date':'2026-06-21T13:04:00','merchant':'BrightMart Groceries','amount':84.30,'currency':'GBP','location':'Manchester, UK','channel':'chip_and_pin','flags':[]},
  'TXN-500022': {'txn_id':'TXN-500022','customer_id':'CUST-88102','card_id':'CARD-3392','date':'2026-06-21T13:05:00','merchant':'BrightMart Groceries','amount':84.30,'currency':'GBP','location':'Manchester, UK','channel':'chip_and_pin','flags':['possible_duplicate']},
  'TXN-500040': {'txn_id':'TXN-500040','customer_id':'CUST-88210','card_id':'CARD-6650','date':'2026-06-19T08:22:00','merchant':'NimbusCloud Ltd','amount':14.99,'currency':'GBP','location':'Online','channel':'card_not_present','flags':['recurring_merchant','customer_unrecognized']},
  'TXN-500077': {'txn_id':'TXN-500077','customer_id':'CUST-88345','card_id':'CARD-1180','date':'2026-06-23T02:47:00','merchant':'LuxGadgetWorld','amount':1890.00,'currency':'USD','location':'Manila, PH','channel':'card_not_present','flags':['foreign_country','high_value','possible_fraud']},
}
KNOWLEDGE = [
  {'title':'Charge Dispute Policy','topic':'disputes','content':'Duplicate charges (same merchant, same amount, within 48 hours) qualify for fast-track provisional credit up to GBP 100 without further review. Unrecognised recurring subscriptions need a 24-hour merchant trace first. Suspected fraud (card-not-present, foreign location, value over GBP 500, or outside the home country) must be escalated to Fraud Operations and the card frozen before any credit. Agent-tier provisional credit is capped at GBP 100.'},
  {'title':'Identity Verification Standard','topic':'security','content':'Verify full name plus one factor (card last 4, date of birth, or phone last 4) before any account action. Never reveal card numbers or balances before verification.'},
  {'title':'Credit Card Products','topic':'credit_cards','content':'Meridian Everyday, Rewards, and Premier cards. Eligibility: UK resident, 18 plus, income 12000 plus. Premier needs income 50000 plus.'},
  {'title':'Personal Loan Products','topic':'loans','content':'Meridian Personal Loan GBP 1000 to 25000, 12 to 60 months, representative APR from 6.4 percent. Soft-search quote with no credit impact.'},
  {'title':'Escalation Model','topic':'governance','content':'Disputes team Priya for compiled dispute packets; Fraud Operations Marcus for suspected fraud; Lending team Sara for credit and loan decisions. Every escalation includes reference id, conversation id, category, confidence, and a compiled summary.'},
]
PRODUCTS = {'credit_cards':[{'product_id':'CC-EVERYDAY','name':'Meridian Everyday','annual_fee_gbp':0,'apr':24.9,'min_income_gbp':12000},{'product_id':'CC-REWARDS','name':'Meridian Rewards','annual_fee_gbp':30,'apr':26.9,'min_income_gbp':15000},{'product_id':'CC-PREMIER','name':'Meridian Premier','annual_fee_gbp':195,'apr':22.9,'min_income_gbp':50000}],'loans':[{'product_id':'LOAN-PERSONAL','name':'Meridian Personal Loan','min_gbp':1000,'max_gbp':25000,'representative_apr':6.4}]}
print('Context loaded:', len(CUSTOMERS), 'customers,', len(TRANSACTIONS), 'transactions,', len(KNOWLEDGE), 'knowledge items')

## Layer 3 - Tools

Tools are how the agent reads context and takes action. Two governance rules are enforced inside the tools so they cannot be skipped: account reads require a prior verify_identity, and state-changing actions require confirmed equals True (the explicit customer yes). Every call writes to the audit trail.

We build the **audit trail first** so every tool can log to it from the first call.

In [ ]:
class AuditTrail:
    def __init__(self):
        self.records = []
    def log(self, conversation_id, action, outcome, category='general', confidence_level='n/a', customer_id=None, location='unknown', status='logged'):
        rec = {'reference_id':'REF-'+uuid.uuid4().hex[:8].upper(),'conversation_id':conversation_id,'timestamp':datetime.now(timezone.utc).isoformat(timespec='seconds'),'customer_id':customer_id,'location':location,'category':category,'confidence_level':confidence_level,'action':action,'outcome':outcome,'status':status}
        self.records.append(rec)
        return rec
    def table(self):
        return pd.DataFrame(self.records)

AUDIT = AuditTrail()
SESSION = {'conversation_id':'CONV-LOCAL','verified':set(),'location':'unknown'}

def start_session(conversation_id):
    SESSION['conversation_id'] = conversation_id
    SESSION['verified'] = set()
    SESSION['location'] = 'unknown'

def log(action, outcome, category='general', confidence_level='n/a', customer_id=None, status='logged'):
    return AUDIT.log(SESSION['conversation_id'], action, outcome, category, confidence_level, customer_id, SESSION['location'], status)

In [ ]:
@tool
def verify_identity(customer_id, name, factor_type, factor_value):
    '''Verify a customer before any account action. Needs name plus one factor: factor_type in card_last4, dob, phone_last4.'''
    c = CUSTOMERS.get(customer_id)
    if not c:
        log('verify_identity','no such customer',category='security',customer_id=customer_id,status='rejected')
        return 'No customer with that id.'
    name_ok = name.strip().lower() == c['name'].lower()
    factor_ok = False
    if factor_type == 'dob':
        factor_ok = factor_value == c['dob']
    elif factor_type == 'phone_last4':
        factor_ok = factor_value == c['phone_last4']
    elif factor_type == 'card_last4':
        factor_ok = any(factor_value == card['last4'] for card in c['cards'])
    if name_ok and factor_ok:
        SESSION['verified'].add(customer_id)
        SESSION['location'] = c['home_location']
        log('verify_identity','verified',category='security',customer_id=customer_id,status='resolved')
        return 'Identity verified for ' + c['name'] + '. You may proceed.'
    log('verify_identity','verification failed',category='security',customer_id=customer_id,status='rejected')
    return 'Verification failed. Do not disclose details or take action.'

@tool
def get_transactions(customer_id):
    '''List recent transactions for a verified customer.'''
    if customer_id not in SESSION['verified']:
        log('get_transactions','blocked: not verified',customer_id=customer_id,status='rejected')
        return 'Customer not verified. Run verify_identity first.'
    txns = [t for t in TRANSACTIONS.values() if t['customer_id']==customer_id]
    log('get_transactions',str(len(txns))+' transactions read',customer_id=customer_id)
    return json.dumps(txns, indent=2)

@tool
def get_transaction(txn_id):
    '''Full details for one transaction, including risk flags.'''
    t = TRANSACTIONS.get(txn_id)
    if not t:
        return 'No such transaction.'
    if t['customer_id'] not in SESSION['verified']:
        log('get_transaction','blocked: not verified',customer_id=t['customer_id'],status='rejected')
        return 'Customer not verified. Run verify_identity first.'
    log('get_transaction','read '+txn_id,customer_id=t['customer_id'])
    return json.dumps(t, indent=2)

@tool
def check_dispute_policy(scenario):
    '''Return dispute policy and thresholds relevant to a scenario.'''
    items = [k for k in KNOWLEDGE if k['topic']=='disputes']
    log('check_dispute_policy','scenario='+scenario,category='charge_dispute')
    return NL.join(k['title']+': '+k['content'] for k in items)

@tool
def search_bank_knowledge(query):
    '''Search Meridian internal knowledge (policies, products, process).'''
    q = query.lower()
    hits = [k for k in KNOWLEDGE if q in (k['title']+' '+k['topic']+' '+k['content']).lower() or any(tok in (k['title']+' '+k['topic']+' '+k['content']).lower() for tok in q.split())]
    log('search_bank_knowledge','query='+query+' hits='+str(len(hits)))
    if not hits:
        return 'No internal knowledge match.'
    return (NL+NL).join(h['title']+': '+h['content'] for h in hits[:3])

In [ ]:
@tool
def raise_dispute(txn_id, reason, confirmed=False):
    '''Raise a formal dispute. Requires a verified customer and confirmed equals True.'''
    t = TRANSACTIONS.get(txn_id)
    if not t:
        return 'No such transaction.'
    if t['customer_id'] not in SESSION['verified']:
        log('raise_dispute','blocked: not verified',category='charge_dispute',customer_id=t['customer_id'],status='rejected')
        return 'Cannot raise a dispute: customer not verified.'
    if not confirmed:
        log('raise_dispute','blocked: not confirmed',category='charge_dispute',customer_id=t['customer_id'],status='rejected')
        return 'Confirmation required. Restate merchant and amount, then call again with confirmed equals True.'
    case_id = 'DSP-'+txn_id.split('-')[-1]
    log('raise_dispute','dispute '+case_id+' opened on '+txn_id,category='charge_dispute',confidence_level='high',customer_id=t['customer_id'],status='resolved')
    return 'Dispute '+case_id+' opened on '+txn_id+' for GBP '+str(t['amount'])+' at '+t['merchant']+'.'

@tool
def issue_provisional_credit(txn_id, amount, confirmed=False):
    '''Issue provisional credit. Agent limit GBP 100. Requires verified plus confirmed equals True.'''
    t = TRANSACTIONS.get(txn_id)
    if not t:
        return 'No such transaction.'
    if t['customer_id'] not in SESSION['verified']:
        return 'Cannot issue credit: customer not verified.'
    if amount > 100:
        log('issue_provisional_credit','blocked: over agent limit',category='charge_dispute',customer_id=t['customer_id'],status='rejected')
        return 'Amount exceeds the GBP 100 agent limit. Escalate to the Disputes team.'
    if not confirmed:
        log('issue_provisional_credit','blocked: not confirmed',category='charge_dispute',customer_id=t['customer_id'],status='rejected')
        return 'Confirmation required. Get an explicit yes, then confirmed equals True.'
    log('issue_provisional_credit','GBP '+str(amount)+' credited for '+txn_id,category='charge_dispute',confidence_level='high',customer_id=t['customer_id'],status='resolved')
    return 'Provisional credit of GBP '+str(amount)+' applied for '+txn_id+'.'

@tool
def freeze_card(card_id, confirmed=False):
    '''Freeze a card to contain suspected fraud. Requires confirmed equals True.'''
    owner = None
    for c in CUSTOMERS.values():
        if any(card['card_id']==card_id for card in c['cards']):
            owner = c
    if not owner:
        return 'No such card.'
    if not confirmed:
        log('freeze_card','blocked: not confirmed',category='charge_dispute',customer_id=owner['customer_id'],status='rejected')
        return 'Confirmation required before freezing a card.'
    log('freeze_card','card '+card_id+' frozen',category='charge_dispute',confidence_level='low',customer_id=owner['customer_id'],status='resolved')
    return 'Card '+card_id+' is now frozen. The Fraud team can issue a replacement.'

@tool
def escalate_to_human(team, summary, reference_id='', confidence_level='n/a'):
    '''Hand a compiled case to a human team: disputes, fraud, or lending.'''
    routes = {'disputes':'Priya (Disputes Lead)','fraud':'Marcus (Fraud Duty Officer)','lending':'Sara (Lending Officer)'}
    owner = routes.get(team.lower())
    if not owner:
        return 'Unknown team. Choose disputes, fraud, or lending.'
    rec = log('escalate_to_human','routed to '+owner+': '+summary[:120],confidence_level=confidence_level,status='escalated')
    return 'Escalated to '+owner+'. Ref '+(reference_id or rec['reference_id'])+', conversation '+SESSION['conversation_id']+', confidence='+confidence_level+'.'

### Scaffolded journeys

Charge disputes are built end to end. Credit card applications and loan enquiries are shown as scaffolds here (catalogue and quote tools). The repo file python/banking_tools.py has the fuller stubs (eligibility, start application, start loan enquiry) to extend the same way.

In [ ]:
@tool
def get_card_products():
    '''List Meridian credit card products.'''
    log('get_card_products','card catalogue read',category='credit_card_application')
    return json.dumps(PRODUCTS['credit_cards'], indent=2)

@tool
def get_loan_products():
    '''List Meridian loan products.'''
    log('get_loan_products','loan catalogue read',category='loan_enquiry')
    return json.dumps(PRODUCTS['loans'], indent=2)

@tool
def estimate_repayment(amount_gbp, term_months, apr):
    '''Indicative monthly loan repayment (soft quote, no credit impact).'''
    r = apr/100/12
    monthly = amount_gbp/term_months if r==0 else amount_gbp*r/(1-(1+r)**(-term_months))
    log('estimate_repayment','GBP '+str(amount_gbp)+' over '+str(term_months)+'m',category='loan_enquiry',confidence_level='high')
    return json.dumps({'monthly_repayment_gbp':round(monthly,2),'note':'Indicative only. Soft search.'}, indent=2)

In [ ]:
TOOLS = [verify_identity, get_transactions, get_transaction, check_dispute_policy, search_bank_knowledge, raise_dispute, issue_provisional_credit, freeze_card, escalate_to_human, get_card_products, get_loan_products, estimate_repayment]
llm_with_tools = llm.bind_tools(TOOLS)
print('Bound', len(TOOLS), 'tools')

### Layer 3 wrap - the system prompt

The system prompt encodes role, brand voice, the three journeys, the verify-before-acting rule, and the confidence routing policy. Editing this changes behaviour without touching code - this is where a strategist does most of the iteration.

In [ ]:
SYSTEM_PROMPT = (
'You are Mia, the AI customer-experience agent for Meridian Bank, a UK retail bank. '
'You help with three journeys: charge disputes, credit card applications, and loan enquiries. '
'You are calm, precise, and never invent facts, balances, fees, or policy. '
'OPERATING LOOP: interpret the goal and category; loop with tools to gather evidence; produce a structured outcome; route through governance before any external action. '
'VERIFY BEFORE ACTING: run verify_identity (name plus one factor) before reading or changing anything. '
'Before any state-changing action (raise_dispute, issue_provisional_credit, freeze_card) restate the action and amount and get an explicit yes; these tools reject calls unless confirmed is true. '
'CONFIDENCE ROUTING: HIGH means resolve at agent level when unambiguous and within limits, for example a clear duplicate under GBP 100; MEDIUM means triage, compile a packet, then escalate; LOW means escalate immediately for suspected fraud, legal, or high ambiguity. When in doubt, route down a tier. '
'DISPUTES: use check_dispute_policy, classify confidence, then act. A duplicate within policy is high: raise_dispute then issue_provisional_credit after confirmation. An unrecognised recurring or ambiguous merchant is medium: compile and escalate to disputes. Suspected fraud is low: offer freeze_card and escalate to fraud. '
'When you have finished using tools, write a short final reply to the customer summarising what you did or what happens next. Always end with a non-empty message. '
'GOVERNANCE: every tool call is logged to an immutable audit trail. Escalations must include reference id, conversation id, category, confidence, and a compiled summary.'
)
system_message = SystemMessage(content=SYSTEM_PROMPT)
print('System prompt set,', len(SYSTEM_PROMPT), 'chars')

## Layer 4 - Orchestration

State, the assistant node, routing, the structured confidence assessment, and the governance handoff. This is the LangGraph part: the assistant decides, the tool node executes, routing decides what happens next, and a final governance node turns the conversation into a structured decision and routes it by confidence.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    conversation_id: str
    category: str
    confidence: float
    confidence_tier: str
    routing: str
    action_taken: str
    resolution_status: str

def assistant(state):
    response = llm_with_tools.invoke([system_message] + state['messages'])
    return {'messages':[response]}

def route_core(state):
    last = state['messages'][-1]
    if getattr(last, 'tool_calls', None):
        return 'tools'
    return 'governance'

tool_node = ToolNode(TOOLS)

### Structured output and confidence routing

At the decision point the model must return a CaseAssessment (category, confidence, recommended action, rationale, escalation team). We map confidence to a tier and a route: high resolves at the agent, medium triages then escalates, low escalates now.

In [ ]:
class CaseAssessment(BaseModel):
    category: str = Field(description='charge_dispute, credit_card_application, loan_enquiry, or other')
    confidence: float = Field(description='0.0 to 1.0 confidence in safely resolving at the agent level')
    recommended_action: str = Field(description='one concrete next action')
    rationale: str = Field(description='why this confidence and action')
    escalation_team: str = Field(default='', description='disputes, fraud, lending, or empty if resolving')

def tier_for(conf):
    if conf >= 0.85:
        return 'high'
    if conf >= 0.5:
        return 'medium'
    return 'low'

ROUTING_RULE = {'high':'resolve_at_agent','medium':'triage_then_escalate','low':'escalate_now'}
assessor = llm.with_structured_output(CaseAssessment)
ASSESSOR_PROMPT = ('You are the governance checkpoint for the Meridian Bank CX agent. Read the conversation and tool evidence and return a CaseAssessment. Be conservative: any sign of fraud, legal risk, or an action above agent limits such as credit over GBP 100 lowers confidence so the case escalates. When in doubt, route down a tier.')

In [ ]:
# The governance checkpoint flattens the conversation into ONE plain-text summary and
# sends it as a single user message. This avoids passing raw tool/assistant turns to the
# structured-output call, which is what causes Anthropic 'messages must have non-empty
# content' / ordering errors. One clean user message in, one structured decision out.

def transcript_of(messages):
    lines = []
    for m in messages:
        role = type(m).__name__
        content = m.content if isinstance(m.content, str) else str(m.content)
        if getattr(m, 'tool_calls', None):
            lines.append(role + ' called tools: ' + ', '.join(tc['name'] for tc in m.tool_calls))
        if content and content.strip():
            lines.append(role + ': ' + content)
    return NL.join(lines) or 'No conversation captured.'

def governance(state):
    transcript = transcript_of(state['messages'])
    a = assessor.invoke([
        SystemMessage(content=ASSESSOR_PROMPT),
        HumanMessage(content='Conversation and tool evidence so far:' + NL + transcript),
    ])
    tier = tier_for(a.confidence)
    routing = ROUTING_RULE[tier]
    AUDIT.log(state['conversation_id'], 'governance_decision', routing + ': ' + a.recommended_action, a.category, tier + ' (' + format(a.confidence, '.2f') + ')', None, SESSION['location'], 'resolved' if tier == 'high' else 'escalated')
    if tier == 'high':
        status = 'Resolved at agent level.'
    elif tier == 'medium':
        status = 'Triaged and escalated to ' + (a.escalation_team or 'human team') + '.'
    else:
        status = 'Escalated immediately to ' + (a.escalation_team or 'human team') + '.'
    return {'category': a.category, 'confidence': a.confidence, 'confidence_tier': tier, 'routing': routing, 'action_taken': a.recommended_action, 'resolution_status': status}

## Layer 5 woven in - build the graph

Governance is not a single node; it runs through every tool (audit) and the final checkpoint. Now we wire the loop: START to assistant; assistant routes to tools or governance; tools back to assistant; governance to END.

`User request -> assistant -> tool call? -> ToolNode -> assistant -> governance handoff -> END`

Note: if you ever re-edit the assistant, tools, or governance functions above, re-run THIS cell so the compiled graph picks up your changes.

In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node('assistant', assistant)
workflow.add_node('tools', tool_node)
workflow.add_node('governance', governance)
workflow.add_edge(START, 'assistant')
workflow.add_conditional_edges('assistant', route_core, {'tools':'tools','governance':'governance'})
workflow.add_edge('tools', 'assistant')
workflow.add_edge('governance', END)
agent = workflow.compile()
print('Agent graph compiled.')

In [ ]:
# Optional: render the graph if graphviz is available
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print('Graph preview skipped:', exc)
    print(agent.get_graph().draw_mermaid())

## Run the agent - three charge-dispute use cases

Each use case exercises a different confidence tier. We pass one customer message per case and let the agent loop through tools, then the governance checkpoint decides the route.

In [ ]:
def run_case(conversation_id, customer_message):
    start_session(conversation_id)
    result = agent.invoke({'messages':[HumanMessage(content=customer_message)],'conversation_id':conversation_id,'category':'','confidence':0.0,'confidence_tier':'','routing':'','action_taken':'','resolution_status':''})
    final = result['messages'][-1].content
    print('CUSTOMER:', customer_message)
    print()
    print('MIA:', final if isinstance(final, str) and final.strip() else '(no text reply)')
    print()
    print('DECISION: category='+result['category']+' tier='+result['confidence_tier']+' ('+format(result['confidence'],'.2f')+') routing='+result['routing'])
    print('OUTCOME:', result['resolution_status'])
    return result

### Use case 1 - Duplicate charge (expected HIGH, resolve at agent)

A clear duplicate within the GBP 100 policy limit. Mia should verify, confirm, raise the dispute, issue provisional credit, and resolve.

In [ ]:
high = run_case('CONV-HIGH-001', 'Hi, I am Daniel Okafor, customer CUST-88102, card ending 9920. I have been charged twice for the same 84.30 BrightMart shop on 21 June, TXN-500021 and TXN-500022. Please sort the duplicate. Yes, I confirm you can raise the dispute and credit me.')

### Use case 2 - Unrecognised subscription (expected MEDIUM, triage then escalate)

A plausible but ambiguous recurring charge. Mia should gather the evidence, compile a packet, and escalate to the Disputes team rather than auto-refund.

In [ ]:
med = run_case('CONV-MED-001', 'Hello, this is Aisha Rahman, CUST-88210, DOB 1995-11-03. There is a 14.99 charge from NimbusCloud Ltd, TXN-500040, that I do not recognise. Can you remove it?')

### Use case 3 - Suspected fraud (expected LOW, escalate now)

A high-value foreign card-not-present charge. Mia should contain (offer to freeze the card) and escalate to Fraud Operations without resolving.

In [ ]:
low = run_case('CONV-LOW-001', 'Hi, George Whitfield here, CUST-88345, card last four 7705. There is a 1890 USD charge from Manila I did not make, TXN-500077. I think my card is compromised.')

## The audit trail (Governance)

Every action across all three conversations, with reference id, timestamp, location, category, confidence, action, and conversation id. This is the artifact compliance and QA review.

In [ ]:
AUDIT.table()

## Inspect the loop

The message trace makes the agent loop visible: assistant decisions and the tools it called for the first case.

In [ ]:
for i, m in enumerate(high['messages']):
    print('---', i, type(m).__name__, '---')
    if getattr(m, 'tool_calls', None):
        print('tool_calls:', [tc['name'] for tc in m.tool_calls])
    if isinstance(getattr(m, 'content', None), str) and m.content:
        print(m.content[:300])
    print()

## Scaffolded journeys - quick look

These run without the LLM, just to show the tool shapes for the other two journeys. Build them out the same way as disputes: add the verify gate, the confirm gate, and a governance route to the Lending team.

In [ ]:
print(get_loan_products.invoke({}))
print(estimate_repayment.invoke({'amount_gbp':10000,'term_months':36,'apr':6.4}))
print(get_card_products.invoke({}))

## Takeaway

You built a banking CX agent that maps onto both frameworks: a model, mocked context, governed tools, a LangGraph orchestration loop with confidence routing, and an immutable audit trail. Disputes are end to end; cards and loans are scaffolded the same way.

**From POC to deployment:** swap the mock dictionaries for real APIs (core banking, card processor, KYC); move the audit trail to an append-only store; replace getpass with a secrets manager; add eval sets per journey and per confidence tier; keep a human in the loop on every medium and low case. The README has the full path.